# Named Entity Recogntion (NER) - XLM-R

## Installing Libraries

In [1]:
!pip install transformers[sentencepiece] datasets
!pip install seqeval
!pip install huggingface-hub
!pip install -q evaluate seqeval
!pip install  --upgrade wandb



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=949f0cf663fdfc1febfc838a6b0cce759354b34eec2505de11f2668a849a1a0a
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.2/27.2 MB 71.8 MB/s eta 0:00:00
  Attempting uninstall: wandb
    Found existing installation: wandb 0.25.0
    Uninstalling wandb-0.25.0:
      Successfully uninstalled wandb-0.25.0


## Importing Libraries

In [2]:
import os

os.environ["WANDB_DISABLED"] = "true"

# import pyarrow
# import datasets

# print(f"✅ Pyarrow version: {pyarrow.__version__}")


import pandas as pd
from collections import defaultdict , Counter
import torch
import torch.nn as nn
from torch.nn.functional import cross_entropy
from datasets import Dataset , DatasetDict , Sequence , Value , Features , ClassLabel
from transformers import AutoTokenizer , XLMRobertaConfig , AutoConfig , TrainingArguments , DataCollatorForTokenClassification , Trainer, AutoModelForTokenClassification, EarlyStoppingCallback
from transformers.modeling_outputs import TokenClassifierOutput
from transformers.models.roberta.modeling_roberta import RobertaModel
from transformers.models.roberta.modeling_roberta import RobertaPreTrainedModel
from seqeval.metrics import f1_score
from sklearn.metrics import ConfusionMatrixDisplay , confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
import evaluate

In [3]:
torch.__version__

'2.10.0+cu128'

In [4]:
import warnings
warnings.filterwarnings('ignore')

In [5]:
# Clone dữ liệu của bài báo VinAI
if not os.path.exists('PhoNER_COVID19'):
    !git clone https://github.com/VinAIResearch/PhoNER_COVID19.git

Cloning into 'PhoNER_COVID19'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 61 (delta 24), reused 41 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 3.61 MiB | 26.61 MiB/s, done.
Resolving deltas: 100% (24/24), done.


In [ ]:
from huggingface_hub import login

## Functions

### Tải dữ liệu PhoNER và Xây dựng Dataset

In [7]:
# Hàm đọc file CoNLL đặc thù của PhoNER
def read_conll_data(file_path):
    tokens, ner_tags = [], []
    current_tokens, current_tags = [], []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_tokens:
                    tokens.append(current_tokens)
                    ner_tags.append(current_tags)
                    current_tokens, current_tags = [], []
            else:
                parts = line.split()
                if len(parts) >= 2:
                    current_tokens.append(parts[0])
                    current_tags.append(parts[-1])
    if current_tokens:
        tokens.append(current_tokens)
        ner_tags.append(current_tags)
    return {"tokens": tokens, "ner_tags": ner_tags}

train_data = read_conll_data("PhoNER_COVID19/data/syllable/train_syllable.conll")
val_data = read_conll_data("PhoNER_COVID19/data/syllable/dev_syllable.conll")
test_data = read_conll_data("PhoNER_COVID19/data/syllable/test_syllable.conll")

# Khởi tạo Label Dictionary
unique_tags = set(tag for doc in train_data["ner_tags"] for tag in doc)
label_list = sorted(list(unique_tags))
if 'O' in label_list: label_list.remove('O')
label_list = ['O'] + label_list

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}

raw_datasets = DatasetDict({
    "train": Dataset.from_dict(train_data),
    "validation": Dataset.from_dict(val_data),
    "test": Dataset.from_dict(test_data)
})

print(f"✅ Đã tải xong tập dữ liệu với {len(label_list)} nhãn thực thể.")

✅ Đã tải xong tập dữ liệu với 21 nhãn thực thể.


### Tokenization (Syllable-level) và Metrics

In [8]:
model_checkpoint = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True, max_length=256)
    labels = []
    for i, tag_sequence in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        new_labels = []
        current_word = None
        for word_id in word_ids:
            if word_id != current_word:
                current_word = word_id
                label_id = -100 if word_id is None else label2id[tag_sequence[word_id]]
                new_labels.append(label_id)
            else:
                new_labels.append(-100)
        labels.append(new_labels)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Dùng num_proc=2 để không bị nghẽn CPU
tokenized_datasets = raw_datasets.map(tokenize_and_align_labels, batched=True, num_proc=2)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

metric = evaluate.load("seqeval")
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [[label_list[p] for (p, l) in zip(pred, lab) if l != -100] for pred, lab in zip(predictions, labels)]
    true_labels = [[label_list[l] for (p, l) in zip(pred, lab) if l != -100] for pred, lab in zip(predictions, labels)]
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "micro_precision": results["overall_precision"],
        "micro_recall": results["overall_recall"],
        "micro_f1": results["overall_f1"]
    }

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map (num_proc=2):   0%|          | 0/5027 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/2000 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/3000 [00:00<?, ? examples/s]

### Huấn luyện XLM-R Base (Tuân thủ Paper)

In [9]:
model_base = AutoModelForTokenClassification.from_pretrained(
    "xlm-roberta-base", num_labels=len(label_list), id2label=id2label, label2id=label2id
)

args_base = TrainingArguments(
    output_dir="xlm-r-base-phoner",
    eval_strategy="epoch", # Từ khóa chuẩn cho thư viện trong notebook này
    push_to_hub=False, # Bật tính năng đẩy lên Hub
    # hub_model_id="Nhocmecode/xlm-r-phoner-uit", # Thay 'your-username' bằng tên HF của bạn
    # hub_strategy="every_save",
    save_strategy="epoch",
    save_only_model=True,
    save_total_limit=1,
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    num_train_epochs=30,
    fp16=True, # Tận dụng GPU P100 tối đa
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    optim="adamw_torch"
)

trainer_base = Trainer(
    model=model_base,
    args=args_base,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

trainer_base.train()
trainer_base.push_to_hub("End of training")

# Đánh giá và lưu bản Base
test_results_base = trainer_base.evaluate(tokenized_datasets["test"])
print("\n--- KẾT QUẢ TẬP TEST (BASE MODEL) ---")
print(f"Micro Precision : {test_results_base['eval_micro_precision']:.4f}")
print(f"Micro Recall    : {test_results_base['eval_micro_recall']:.4f}")
print(f"Micro F1-Score: {test_results_base['eval_micro_f1']:.4f}")
trainer_base.save_model("./best_model_phoner_base")

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Micro Precision,Micro Recall,Micro F1
1,No log,0.245255,0.843715,0.907462,0.874428
2,No log,0.167765,0.892384,0.937015,0.914155
3,No log,0.143326,0.920886,0.951056,0.935728
4,No log,0.157504,0.933237,0.949585,0.941340
5,No log,0.178073,0.930459,0.946510,0.938416
6,No log,0.173860,0.934709,0.957208,0.945825
7,0.223853,0.177910,0.934242,0.953731,0.943886
8,0.223853,0.190553,0.941068,0.956673,0.948806
9,0.223853,0.190553,0.942178,0.954400,0.948250
10,0.223853,0.194351,0.944393,0.953865,0.949105


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


--- KẾT QUẢ TẬP TEST (BASE MODEL) ---
Micro Precision : 0.9395
Micro Recall    : 0.9484
Micro F1-Score: 0.9439


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### Dọn dẹp RAM & GPU

In [10]:
import gc
import torch
del trainer_base, model_base
gc.collect()
torch.cuda.empty_cache()
print("Đã giải phóng GPU, sẵn sàng cho bản Large!")

Đã giải phóng GPU, sẵn sàng cho bản Large!


### Huấn luyện XLM-R Large (Với Gradient Accumulation)

In [11]:
model_large = AutoModelForTokenClassification.from_pretrained(
    "xlm-roberta-large", num_labels=len(label_list), id2label=id2label, label2id=label2id
)

args_large = TrainingArguments(
    output_dir="xlm-r-large-phoner",
    push_to_hub=False, # Bật tính năng đẩy lên Hub
    # hub_model_id="Nhocmecode/xlm-r-phoner-uit", # Thay 'your-username' bằng tên HF của bạn
    # hub_strategy="every_save",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_only_model=True,
    save_total_limit=1,
    learning_rate=5e-5,
    per_device_train_batch_size=8,   
    gradient_accumulation_steps=4, # 8 * 4 = 32 (Khớp chuẩn bài báo)
    num_train_epochs=30,
    fp16=True,                       
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    optim="adamw_torch"
)

trainer_large = Trainer(
    model=model_large,
    args=args_large,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

trainer_large.train()
trainer_large.push_to_hub("End of training")

# Đánh giá kết quả cuối cùng
test_results_large = trainer_large.evaluate(tokenized_datasets["test"])
print("\n--- KẾT QUẢ TẬP TEST (LARGE MODEL) ---")
print(f"Micro Precision : {test_results_large['eval_micro_precision']:.4f}")
print(f"Micro Recall    : {test_results_large['eval_micro_recall']:.4f}")
print(f"Micro F1-Score  : {test_results_large['eval_micro_f1']:.4f}")
trainer_large.save_model("./best_model_phoner_large")

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Micro Precision,Micro Recall,Micro F1
1,No log,0.208527,0.858455,0.937550,0.896261
2,No log,0.173362,0.903642,0.935544,0.919317
3,No log,0.147089,0.923839,0.947312,0.935428
4,No log,0.170001,0.938392,0.949184,0.943757
5,No log,0.178783,0.921282,0.945306,0.933140
6,No log,0.176277,0.936406,0.958946,0.947542
7,0.736313,0.173848,0.940967,0.952795,0.946844
8,0.736313,0.193987,0.939966,0.958946,0.949361
9,0.736313,0.189223,0.939263,0.957475,0.948282
10,0.736313,0.208363,0.942341,0.955068,0.948662


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


--- KẾT QUẢ TẬP TEST (LARGE MODEL) ---
Micro Precision : 0.9527
Micro Recall    : 0.9536
Micro F1-Score  : 0.9532


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]